# Módulo 01 · Aula 4: Carregando documentos

**Duração:** 45 min

## Objetivos de aprendizagem

Ao final desta aula você será capaz de:

- explicar o que é um `Document` (`page_content` + `metadata`) e por que os **metadados** importam em RAG;
- carregar os quatro formatos da base: **Markdown** com front matter, **PDF** página a página com `pypdf`, **HTML**
  só com o artigo com `BeautifulSoup` e **CSV** linha a linha;
- reconhecer problemas reais de ingestão: PDF sem texto, HTML com lixo e encoding errado;
- usar a seção **1. Carregar** de `src/rag/ingestion.py` e inventariar a base com `load_folder()`.

## Pré-requisitos

- Aulas 1.1 a 1.3: embeddings, busca por similaridade e o índice FAISS de `src/rag/vectorstore.py`.
- Python: `pathlib`, dicionários e pandas básico.

## O que vamos construir

Nesta aula e na próxima abrimos a caixa da **ingestão**, o caminho dos arquivos de `data/documents/` até o índice.

> A qualidade do RAG começa na ingestão. Documento bem carregado, bem dividido e com bons metadados é metade da
> resposta certa.

Esta aula cuida da primeira etapa: transformar arquivos de formatos diferentes em uma lista uniforme de `Document`.

```mermaid
flowchart LR
    MD[policies/*.md<br/>internal/*.md] --> L1[load_markdown]
    PDF[manuals/*.pdf] --> L2[load_pdf]
    HTML[help_center/*.html] --> L3[load_html]
    CSV[faq.csv] --> L4[load_csv]
    L1 & L2 & L3 & L4 --> D["list[Document]"]
    D -.aula 1.5.-> S[dividir]
    S -.aula 1.5.-> I[(índice)]
```

In [1]:
# Os notebooks rodam a partir da raiz do projeto: é lá que ficam o .env, a pasta data/ e o pacote src/.
import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Chat e embeddings vêm do Azure OpenAI configurado no .env (veja src/config.py).
from src.config import settings

## 1. O `Document`: texto e metadados

Todo componente de RAG do LangChain (splitters, vector stores, retrievers) conversa pelo mesmo tipo,
`langchain_core.documents.Document`, com dois campos:

- `page_content`: o **texto**, que vira vetor e que o modelo vai ler;
- `metadata`: um dicionário com **tudo o que não é o texto**: de onde veio, página, seção, se está em vigor...

Os metadados **não viram vetor**, mas fazem duas coisas que o vetor sozinho não faz: **filtrar** a busca (nunca
mostrar ao cliente uma política arquivada ou um procedimento interno) e **citar** a fonte ("Política de Troca, seção
Resumo da regra"). Um exemplo só com Python, usando trechos reais da base:

In [2]:
from langchain_core.documents import Document

examples = [
    Document(
        page_content="Trocas podem ser solicitadas em até 30 dias corridos após a entrega do pedido.",
        metadata={"source": "policies/exchange.md", "type": "markdown", "title": "Política de Troca",
                  "section": "Resumo da regra", "status": "active", "audience": "customers"},
    ),
    Document(
        page_content="Frete grátis para compras acima de R$ 199,00, em todo o Brasil.",
        metadata={"source": "policies/shipping_2024.md", "title": "Política de Frete e Entrega",
                  "status": "archived", "audience": "customers"},
    ),
    Document(
        page_content="Reembolsos de R$ 5.000,01 a R$ 15.000,00 são aprovados pelo supervisor de atendimento.",
        metadata={"source": "internal/refund_approval_procedure.md", "title": "Procedimento Interno de Aprovação de Reembolsos",
                  "status": "active", "audience": "internal"},
    ),
]


def allowed_for_customers(document: Document) -> bool:
    """Só conteúdo em vigor e destinado a clientes pode chegar à resposta."""
    return document.metadata["status"] == "active" and document.metadata["audience"] == "customers"


def citation(document: Document) -> str:
    meta = document.metadata
    parts = [meta["title"], meta.get("section"), f"página {meta['page']}" if meta.get("page") else None]
    return " > ".join(p for p in parts if p) + f" ({meta['source']})"


for document in examples:
    status = "entra" if allowed_for_customers(document) else "BLOQUEADO"
    print(f"{status:<10} {citation(document)}")

entra      Política de Troca > Resumo da regra (policies/exchange.md)
BLOQUEADO  Política de Frete e Entrega (policies/shipping_2024.md)
BLOQUEADO  Procedimento Interno de Aprovação de Reembolsos (internal/refund_approval_procedure.md)


**Observe:** as três frases poderiam aparecer numa busca sobre frete ou reembolso; só os metadados dizem que a regra de
R$ 199 é antiga e que o procedimento é interno. Se o carregador não gravar `status` e `audience`, nenhuma etapa
posterior recupera essa informação.

| Chave | Exemplo | Para que serve |
|---|---|---|
| `source` | `manuals/fn-001_nova_buds.pdf` | citação e atualização do índice (remover ou reindexar um arquivo) |
| `type` | `markdown`, `pdf`, `html`, `faq` | escolher como dividir; estatísticas |
| `title`, `section`, `subsection` | `Política de Troca`, `Resumo da regra` | citação e cabeçalho de contexto (aula 1.5) |
| `page` / `row` | `3` / `12` | citar a página do PDF ou a linha do CSV |
| `status` / `audience` | `active`, `archived` / `customers`, `internal` | filtros: só o que está em vigor e o que o cliente pode ver |
| `updated_at` | `2026-06-15` | saber se o conteúdo está atualizado |

> 💡 **Dica:** um bom teste para decidir se algo é metadado: "vou precisar filtrar, ordenar ou citar por isso?". Se
> sim, grave no carregador.

## 2. Markdown com front matter

As políticas e o procedimento interno estão em Markdown, com um **front matter** no topo: um bloco entre `---` com
pares `chave: valor`. O front matter vira `metadata` e o resto vira `page_content`. Com `split("---", 2)`, o texto se
divide em três partes: o que vem antes do primeiro `---` (vazio), o bloco de metadados e o corpo.

In [3]:
from pathlib import Path

DOCS = Path(settings.documents_dir)  # data/documents

exchange_raw = (DOCS / "policies/exchange.md").read_text(encoding="utf-8")
print("\n".join(exchange_raw.splitlines()[:12]))
print("...\n" + "=" * 100)

_, front_matter, body = exchange_raw.split("---", 2)
metadata = {}
for line in front_matter.strip().splitlines():
    key, value = line.split(":", 1)
    metadata[key.strip()] = value.strip()

exchange_doc = Document(
    page_content=body.strip(),
    metadata={**metadata, "source": "policies/exchange.md", "type": "markdown"},
)
print(exchange_doc.metadata)
print(f"\n{len(exchange_doc.page_content)} caracteres, começando por:\n{exchange_doc.page_content[:230]}...")

---
title: Política de Troca
category: policy
updated_at: 2026-06-15
version: 4
status: active
audience: customers
---

# Política de Troca

A TechNova é uma loja exclusivamente online e quer que você fique satisfeito com cada compra. Esta política explica quando e como trocar um produto comprado no site ou no app, quais condições precisam ser atendidas e quanto tempo cada etapa leva.
...
{'title': 'Política de Troca', 'category': 'policy', 'updated_at': '2026-06-15', 'version': '4', 'status': 'active', 'audience': 'customers', 'source': 'policies/exchange.md', 'type': 'markdown'}

5256 caracteres, começando por:
# Política de Troca

A TechNova é uma loja exclusivamente online e quer que você fique satisfeito com cada compra. Esta política explica quando e como trocar um produto comprado no site ou no app, quais condições precisam ser aten...


**Observe:** `status` e `audience` vêm do front matter, que é onde a equipe de conteúdo marca o que está em vigor e para
quem. Como vimos no tour da aula 1.1, `policies/shipping_2024.md` está `archived` e o procedimento de reembolso é
`internal`: os dois continuam na base de propósito, e o filtro decide o que o cliente vê.

> ⚠️ O corpo mantém as marcas `#`, `**` e `-`: elas ajudam o modelo a entender a estrutura e, na aula 1.5, os títulos
> `##` e `###` serão usados para dividir o texto por seção.

## 3. PDF página a página com `pypdf`

Os manuais estão em PDF, um formato de **desenho de página** (posições de letras, linhas, imagens), como mostram os
primeiros bytes. O `pypdf` reconstrói o texto de cada página e lê os metadados do próprio PDF. Criamos **um
`Document` por página**, com o número da página nos metadados, para a resposta poder citar "Manual do Nova Buds,
página 3".

In [4]:
from pypdf import PdfReader

buds_path = DOCS / "manuals/fn-001_nova_buds.pdf"
print(buds_path.read_bytes()[:60], "\n")

reader = PdfReader(buds_path)
for key, value in reader.metadata.items():
    print(f"  {key:<14} {value}")

pdf_docs = [
    Document(
        page_content=page.extract_text(),
        metadata={
            "source": buds_path.relative_to(DOCS).as_posix(),
            "type": "pdf",
            "title": reader.metadata.title,
            "product": reader.metadata.subject,
            "page": number,
        },
    )
    for number, page in enumerate(reader.pages, start=1)
]

print()
for d in pdf_docs:
    first_lines = " | ".join(d.page_content.splitlines()[1:3])
    print(f"página {d.metadata['page']}: {len(d.page_content):>5} caracteres  {first_lines[:70]}")

# O texto de uma página por dentro: é aí que aparecem as limitações da extração.
print("\n" + "=" * 100)
page_3 = pdf_docs[2].page_content
print(page_3[:700])
print("...")
print(page_3[-160:])

b'%PDF-1.3\n%\xe9\xeb\xf1\xbf\n1 0 obj\n<<\n/Count 5\n/Kids [3 0 R\n5 0 R\n7 0 R\n' 

  /Author        TechNova
  /CreationDate  D:20260927230838Z
  /Creator       TechNova Documentação Técnica
  /Keywords      FN-001, Fone Nova Buds, manual, TechNova
  /Subject       SKU FN-001 | Fone Nova Buds
  /Title         Manual do usuário: Fone Nova Buds

página 1:   449 caracteres  Fone Nova Buds | SKU: FN-001
página 2:  2064 caracteres  1. Apresentação | Obrigado por escolher o Fone Nova Buds (SKU FN-001),
página 3:  2127 caracteres  4. Primeiros passos | 4.1 Primeira carga
página 4:  2563 caracteres  LED Significado | Pulsando lentamente com o cabo
página 5:  3186 caracteres  Problema Causa provável Solução | Só um fone toca Fone sem carga ou

TechNova | Fone Nova Buds (FN-001) | Manual do usuário
4. Primeiros passos
4.1 Primeira carga
Retire as películas de proteção dos contatos, coloque os fones no estojo e carregue até o LED ficar verde fixo
antes do primeiro uso.
4.2 Pareamento
- Abra o 

**Observe:**

- Toda página começa com o cabeçalho corrido `TechNova | Fone Nova Buds (FN-001) | Manual do usuário` e termina com
  `Página 3 de 5`. É ruído útil: o nome do produto no topo ajuda a busca (aula 1.5).
- A tabela de controles por toque virou texto corrido ("Gesto Função... 1 toque..."): a extração não preserva o
  layout. Para tabelas complexas existem extratores de layout, fora do escopo do curso.
- As quebras de linha seguem a largura da página, não as frases.

## 4. HTML: só o `<article>`, sem menu e rodapé

Os artigos da central de ajuda são páginas HTML completas, com `<head>`, busca, menu, trilha de navegação e rodapé
em volta do `<article>`. Veja o começo do arquivo cru e compare, com `BeautifulSoup`, o texto da página inteira
(`<body>`) com o do artigo:

In [5]:
from bs4 import BeautifulSoup

track_path = DOCS / "help_center/track_order.html"
track_raw = track_path.read_text(encoding="utf-8")
print("\n".join(track_raw.splitlines()[:26]))
print("...\n" + "=" * 100)

soup = BeautifulSoup(track_raw, "html.parser")
body_text = soup.body.get_text("\n", strip=True)
article_text = soup.find("article").get_text("\n", strip=True)
print(f"<body>: {len(body_text)} caracteres | <article>: {len(article_text)} caracteres")

article_lines = set(article_text.splitlines())
junk = [line for line in body_text.splitlines() if line not in article_lines]
print(f"{len(junk)} linhas só existem fora do artigo, por exemplo: {junk[-6:-2]}")

# O mesmo rodapé se repete nos outros artigos?
footer_phrase = "Formas de pagamento: Pix, cartão de crédito e boleto bancário."
help_files = sorted((DOCS / "help_center").glob("*.html"))
with_phrase = [
    p.name for p in help_files
    if footer_phrase in BeautifulSoup(p.read_text(encoding="utf-8"), "html.parser").body.get_text()
]
print(f"A frase do rodapé aparece em {len(with_phrase)} de {len(help_files)} artigos.")

<!doctype html>
<html lang="pt-BR">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Como rastrear seu pedido | Central de Ajuda TechNova</title>
  <meta name="description" content="Saiba onde encontrar o código de rastreio, como identificar a transportadora e o que significa cada status do seu pedido TechNova.">
  <meta name="updated_at" content="2026-08-12">
  <meta name="robots" content="index, follow">
  <link rel="stylesheet" href="/assets/css/central-ajuda.css">
  <script src="/assets/js/analytics.js" defer></script>
</head>
<body>
  <header class="topo">
    <a class="logo" href="/">TechNova</a>
    <form class="busca" action="/ajuda/busca" method="get">
      <label for="q">Buscar na Central de Ajuda</label>
      <input id="q" name="q" type="search" placeholder="Como podemos ajudar?">
      <button type="submit">Buscar</button>
    </form>
  </header>
  <nav class="menu-principal" aria-label="Menu principal">
    <

Parece pouco (cerca de 15% do texto), mas o problema é a **repetição**: o mesmo menu e o mesmo rodapé estão em
**todos** os artigos. No índice, uma pergunta como "quais as formas de pagamento?" encontraria o **rodapé** de 16
artigos que não tratam de pagamento, e os links "Política de troca, Política de garantia..." aproximariam todo artigo
de qualquer pergunta sobre políticas.

Montamos o `Document` só com o artigo, aproveitando `<title>` e as tags `<meta>` como metadados:

In [6]:
description = soup.find("meta", attrs={"name": "description"})["content"]
updated_at = soup.find("meta", attrs={"name": "updated_at"})["content"]

html_doc = Document(
    page_content=article_text,
    metadata={
        "source": track_path.relative_to(DOCS).as_posix(),
        "type": "html",
        "title": soup.title.get_text(strip=True),
        "description": description,
        "updated_at": updated_at,
    },
)
print(html_doc.metadata)
print()
print(html_doc.page_content[:400])

{'source': 'help_center/track_order.html', 'type': 'html', 'title': 'Como rastrear seu pedido | Central de Ajuda TechNova', 'description': 'Saiba onde encontrar o código de rastreio, como identificar a transportadora e o que significa cada status do seu pedido TechNova.', 'updated_at': '2026-08-12'}

Como rastrear seu pedido
Depois que o pagamento é confirmado, você pode acompanhar cada etapa do seu pedido pelo site ou pelo app TechNova. Neste artigo explicamos onde encontrar o código de rastreio, como saber qual transportadora está com a sua encomenda e o que fazer se o rastreio não mostrar novidades.
Onde encontrar o código de rastreio
O código de rastreio é gerado no momento em que o pedido


> 💡 **Dica:** cada site tem a sua estrutura. Antes de escrever o carregador, abra dois ou três arquivos e descubra qual
> elemento guarda o conteúdo (`<article>`, `<main>`, uma `div` específica).

## 5. CSV: uma linha, um `Document`

As perguntas frequentes estão em `faq.csv` (`question`, `answer`, `category`). A resposta da linha 3 tem vírgulas e
até aspas dentro dela (`""Cancelar pedido""`): por isso **nunca** se lê CSV com `line.split(",")`. O módulo `csv`
trata aspas e vírgulas corretamente, e cada linha (uma pergunta com a sua resposta) vira um `Document` inteiro.

In [7]:
import csv

faq_path = DOCS / "faq.csv"
line_3 = faq_path.read_text(encoding="utf-8").splitlines()[2]
print("Linha 3 crua:", line_3[:120], "...")
print("split(',') na linha 3 devolve", len(line_3.split(",")), "campos (o arquivo tem 3 colunas)\n")

with faq_path.open(encoding="utf-8", newline="") as file:
    faq_docs = [
        Document(
            page_content=f"Pergunta: {row['question']}\nResposta: {row['answer']}",
            metadata={"source": "faq.csv", "type": "faq", "category": row["category"], "row": number},
        )
        for number, row in enumerate(csv.DictReader(file), start=2)  # a linha 1 é o cabeçalho
    ]

print(f"{len(faq_docs)} perguntas frequentes. A da linha 3:")
print(faq_docs[1].page_content)
print(faq_docs[1].metadata)

Linha 3 crua: Posso cancelar meu pedido?,"Sim, sem custo, enquanto ele estiver com status processando, ou seja, antes do despacho. Use ...
split(',') na linha 3 devolve 8 campos (o arquivo tem 3 colunas)

40 perguntas frequentes. A da linha 3:
Pergunta: Posso cancelar meu pedido?
Resposta: Sim, sem custo, enquanto ele estiver com status processando, ou seja, antes do despacho. Use o botão "Cancelar pedido" em Minha Conta > Meus Pedidos ou fale com o atendimento. Depois do envio não é possível cancelar, mas você pode recusar a entrega ou pedir a devolução em até 30 dias após receber.
{'source': 'faq.csv', 'type': 'faq', 'category': 'orders', 'row': 3}


**Observe:** o texto junta pergunta e resposta com rótulos: a pergunta ajuda a busca (clientes perguntam de jeitos
parecidos) e a resposta é o que o modelo vai usar. `row` é a linha do arquivo, útil para achar e corrigir uma FAQ.

| Formato | Pasta | Biblioteca | Unidade | O que fica de fora |
|---|---|---|---|---|
| Markdown | `policies/`, `internal/` | leitura de texto | 1 documento por arquivo | o bloco `---` (vira metadado) |
| PDF | `manuals/` | `pypdf` | 1 documento por página | layout e páginas sem texto |
| HTML | `help_center/` | `BeautifulSoup` | 1 documento por artigo | head, menu, trilha, rodapé, scripts |
| CSV | `faq.csv` | `csv` | 1 documento por linha | o cabeçalho (vira nomes de campos) |

## 6. Problemas reais de ingestão

A base da TechNova é bem comportada; bases reais não são. Simulamos três problemas comuns em uma pasta temporária
(nada em `data/documents` é alterado).

### 6.1 PDF sem texto (digitalizado)

Um PDF escaneado é uma **imagem** de cada página: não há letras para extrair. Simulamos com páginas em branco, o mesmo
resultado que o `pypdf` teria com uma digitalização, e verificamos todos os manuais da base:

In [8]:
import tempfile

from pypdf import PdfWriter

WORK_DIR = Path(tempfile.mkdtemp(prefix="technova_01_04_"))

writer = PdfWriter()
for _ in range(3):
    writer.add_blank_page(width=595, height=842)  # A4 em pontos
scanned_path = WORK_DIR / "scanned_manual.pdf"
with scanned_path.open("wb") as file:
    writer.write(file)


def pages_without_text(path: Path) -> list[int]:
    """Números das páginas de onde o pypdf não conseguiu extrair texto."""
    return [n for n, page in enumerate(PdfReader(path).pages, start=1) if not (page.extract_text() or "").strip()]


for path in [scanned_path, *sorted((DOCS / "manuals").glob("*.pdf"))]:
    empty = pages_without_text(path)
    print(f"{path.name:<28} {'OK' if not empty else f'SEM TEXTO nas páginas {empty}'}")

scanned_manual.pdf           SEM TEXTO nas páginas [1, 2, 3]
cx-001_nova_boom.pdf         OK
fn-001_nova_buds.pdf         OK
fn-002_nova_studio.pdf       OK
mn-001_nova_view_27.pdf      OK
nb-001_nova_air_14.pdf       OK
nb-002_nova_pro_16.pdf       OK
sm-001_nova_x.pdf            OK
sm-004_nova_mini.pdf         OK


Nenhum erro, nenhum aviso: só texto vazio, o pior tipo de problema, porque passa em silêncio. O carregador do projeto
pula páginas sem texto, então vale **detectar e avisar**. A solução para PDFs digitalizados é **OCR** (como o
Tesseract) ou um serviço de extração de documentos, antes da ingestão.

### 6.2 HTML sem `<article>` e com lixo

Nem toda página tem um `<article>`. O carregador do projeto procura o artigo e, se não houver, usa o `<body>`,
removendo sempre `script`, `style`, `nav` e `footer`. Testando com uma página sem `<article>`:

In [9]:
from src.rag.ingestion import load_html

messy_html = """<html><head><title>Horário de atendimento</title><style>p {color: red}</style></head>
<body>
  <nav><a href="/">Início</a> <a href="/ofertas">Ofertas</a></nav>
  <h1>Horário de atendimento</h1>
  <p>O chat funciona de segunda a sábado, das 8h às 20h.</p>
  <script>trackPageView("atendimento");</script>
  <footer>© 2026 TechNova. Todos os direitos reservados.</footer>
</body></html>"""
messy_path = WORK_DIR / "no_article.html"
messy_path.write_text(messy_html, encoding="utf-8")

print("Texto bruto do <body>:")
print(BeautifulSoup(messy_html, "html.parser").body.get_text(" ", strip=True))
print("\nload_html():")
print(load_html(messy_path, WORK_DIR)[0].page_content)

Texto bruto do <body>:
Início Ofertas Horário de atendimento O chat funciona de segunda a sábado, das 8h às 20h. © 2026 TechNova. Todos os direitos reservados.

load_html():
Horário de atendimento

O chat funciona de segunda a sábado, das 8h às 20h.


### 6.3 Encoding

O **encoding** diz como transformar bytes em letras. A base usa UTF-8, mas arquivos exportados de sistemas antigos (ou
do Excel no Windows) costumam vir em Latin-1 (cp1252). Com acentos, isso quebra. No fim, carregamos a pasta temporária
inteira com `load_folder`, com os três problemas juntos:

In [10]:
import shutil

from src.rag.ingestion import load_folder

text = "---\ntitle: Política de Devolução\nstatus: active\n---\n\nA devolução é gratuita em até 7 dias."
latin1_path = WORK_DIR / "latin1_policy.md"
latin1_path.write_bytes(text.encode("cp1252"))
utf8_path = WORK_DIR / "utf8_policy.md"
utf8_path.write_text(text, encoding="utf-8")

try:
    latin1_path.read_text(encoding="utf-8")
except UnicodeDecodeError as error:
    print("Latin-1 lido como UTF-8:", error)
# O erro contrário não levanta exceção: bytes UTF-8 lidos como Latin-1 viram "mojibake".
print("UTF-8 lido como Latin-1:", utf8_path.read_text(encoding="latin-1").splitlines()[1], "\n")

# A pasta inteira: load_folder registra no log o arquivo que falhou e segue com os outros.
loaded = load_folder(WORK_DIR)
print("\nCarregados:", [d.metadata["source"] for d in loaded])

# A correção: sabemos a origem (cp1252), então convertemos o arquivo para UTF-8 uma única vez.
latin1_path.write_text(latin1_path.read_bytes().decode("cp1252"), encoding="utf-8")
print("Depois de converter:", latin1_path.read_text(encoding="utf-8").splitlines()[1])

shutil.rmtree(WORK_DIR)  # limpeza da pasta temporária da aula

Latin-1 lido como UTF-8: 'utf-8' codec can't decode byte 0xed in position 14: invalid continuation byte
UTF-8 lido como Latin-1: title: PolÃ­tica de DevoluÃ§Ã£o 

2026-09-27 22:58:52 | WARNING  | src.rag.ingestion | Falha ao carregar | arquivo=latin1_policy.md | erro=UnicodeDecodeError: 'utf-8' codec can't decode byte 0xed in position 14: invalid continuation byte


2026-09-27 22:58:52 | INFO     | src.rag.ingestion | Documentos carregados | arquivos=4 | documentos=2



Carregados: ['no_article.html', 'utf8_policy.md']
Depois de converter: title: Política de Devolução


**Observe:** dos quatro arquivos da pasta temporária, só dois viraram `Document`. O PDF digitalizado não gerou nada,
**sem erro nenhum**; o arquivo em Latin-1 falhou, e o `load_folder` registrou o erro no log (com o nome do arquivo) e
seguiu com os outros. O mojibake é o pior caso: "DevoluÃ§Ã£o" entraria no índice sem erro e nunca seria encontrado
por quem busca "devolução". Regras práticas:

- declare o encoding sempre (`read_text(encoding="utf-8")`) e converta os arquivos para UTF-8 **uma vez**, na origem;
- procure sinais de mojibake (`Ã`, `Â`) no texto carregado;
- resiliência não é esconder erros: conte os arquivos que falharam e trate isso como alerta. Um manual que não entrou
  no índice é uma pergunta que o assistente não vai saber responder.

## 7. E os document loaders do LangChain?

Os *document loaders* prontos (`PyPDFLoader`, `BSHTMLLoader`, `CSVLoader`, `UnstructuredMarkdownLoader`...) vivem no
`langchain-community`, em **descontinuação**. O projeto escreve os carregadores sobre as mesmas bibliotecas que eles
usavam por baixo (`pypdf`, `BeautifulSoup`, `csv`), em cerca de 100 linhas, e ganha **controle**: quais metadados cada
formato grava, com nomes padronizados, e o que é descartado. A única dependência do community que sobra no projeto é o
FAISS, isolada em `src/rag/vectorstore.py` (aula 1.3).

## 8. Do notebook para o projeto: [src/rag/ingestion.py](../src/rag/ingestion.py)

A seção **1. Carregar** do arquivo reúne o que fizemos, com alguns cuidados a mais:

| Função | O que faz |
|---|---|
| `parse_front_matter(text)` | separa o bloco `---` do corpo do Markdown; tolera arquivos sem front matter |
| `load_markdown`, `load_pdf`, `load_html`, `load_csv` | um carregador por formato, como nas seções 2 a 5; o de PDF pula páginas sem texto |
| `LOADERS` / `load_file(path, base)` | dicionário extensão -> carregador / escolhe o carregador pela extensão |
| `load_folder(folder)` | carrega a pasta inteira (e subpastas) em ordem alfabética, registrando as falhas no log |

Decisões de design:

- **`DEFAULT_METADATA`**: todo documento sai com `status="active"` e `audience="customers"`, a menos que o arquivo
  diga outra coisa. Assim o filtro do módulo 02 nunca encontra uma chave faltando.
- **`source` relativo à pasta base** (`policies/exchange.md`), com barras normais: o mesmo valor em qualquer máquina, o
  que é essencial para atualizar o índice (aula 1.5).
- **`_clean`** remove espaços repetidos e linhas em branco em excesso; todos os arquivos são lidos com
  `encoding="utf-8"` explícito.

Um arquivo de cada formato com `load_file`, conferindo que o produto chega ao mesmo texto que montamos à mão e que um
formato desconhecido gera um erro claro:

In [11]:
from src.rag.ingestion import DEFAULT_METADATA, LOADERS, load_file, parse_front_matter

print("Formatos suportados:", {ext: fn.__name__ for ext, fn in LOADERS.items()})
print("Metadados padrão:", DEFAULT_METADATA)
print()
for relative in ["policies/exchange.md", "manuals/fn-001_nova_buds.pdf", "help_center/track_order.html", "faq.csv"]:
    loaded = load_file(DOCS / relative, base=DOCS)
    print(f"{relative}: {len(loaded)} Document(s)")
    print("   ", loaded[0].metadata)

print()
product_md = load_file(DOCS / "policies/exchange.md", base=DOCS)[0]
product_pdf = load_file(buds_path, base=DOCS)
print("Markdown igual ao nosso:", product_md.page_content == exchange_doc.page_content)
print("Front matter igual ao nosso:", parse_front_matter(exchange_raw)[0] == metadata)
print("PDF: páginas", len(product_pdf), "| caracteres (nosso x produto):",
      sum(len(d.page_content) for d in pdf_docs), "x", sum(len(d.page_content) for d in product_pdf))

try:
    load_file("contract.docx")
except ValueError as error:
    print("\nValueError:", error)

Formatos suportados: {'.md': 'load_markdown', '.pdf': 'load_pdf', '.html': 'load_html', '.csv': 'load_csv'}
Metadados padrão: {'status': 'active', 'audience': 'customers'}

policies/exchange.md: 1 Document(s)
    {'status': 'active', 'audience': 'customers', 'title': 'Política de Troca', 'category': 'policy', 'updated_at': '2026-06-15', 'version': '4', 'source': 'policies/exchange.md', 'type': 'markdown'}
manuals/fn-001_nova_buds.pdf: 5 Document(s)
    {'status': 'active', 'audience': 'customers', 'source': 'manuals/fn-001_nova_buds.pdf', 'type': 'pdf', 'category': 'manual', 'title': 'Manual do usuário: Fone Nova Buds', 'product': 'SKU FN-001 | Fone Nova Buds', 'page': 1, 'total_pages': 5}
help_center/track_order.html: 1 Document(s)
    {'status': 'active', 'audience': 'customers', 'source': 'help_center/track_order.html', 'type': 'html', 'category': 'help_center', 'title': 'Como rastrear seu pedido | Central de Ajuda TechNova', 'description': 'Saiba onde encontrar o código de rastreio

Agora a base inteira com `load_folder()`, agrupada por tipo, com os documentos fora do padrão (status ou público):

In [12]:
import pandas as pd

documents = load_folder()  # padrão: settings.documents_dir

inventory = pd.DataFrame(
    [
        {"source": d.metadata["source"], "type": d.metadata["type"], "status": d.metadata["status"],
         "audience": d.metadata["audience"], "chars": len(d.page_content)}
        for d in documents
    ]
)
by_type = inventory.groupby("type").agg(
    files=("source", "nunique"), documents=("source", "size"),
    min_chars=("chars", "min"), mean_chars=("chars", "mean"), max_chars=("chars", "max"),
)
print(by_type.round(0).astype(int).to_string())

by_file = inventory.groupby(["source", "type", "status", "audience"], as_index=False).agg(chars=("chars", "sum"))
print("\nFora do padrão (status ou público):")
print(by_file[(by_file["status"] != "active") | (by_file["audience"] != "customers")].to_string(index=False))
total_chars = f"{inventory['chars'].sum():_}".replace("_", ".")
print(f"\n{len(by_file)} arquivos, {len(documents)} documentos, {total_chars} caracteres")

2026-09-27 22:58:53 | INFO     | src.rag.ingestion | Documentos carregados | arquivos=34 | documentos=105


          files  documents  min_chars  mean_chars  max_chars
type                                                        
faq           1         40        132         234        345
html         16         16       3339        3689       4101
markdown      9          9       3490        4635       5411
pdf           8         40        449        1993       3186

Fora do padrão (status ou público):
                               source     type   status  audience  chars
internal/refund_approval_procedure.md markdown   active  internal   5297
            policies/shipping_2024.md markdown archived customers   3490

34 arquivos, 105 documentos, 189.805 caracteres


**Observe:**

- 34 arquivos viraram 105 documentos: 9 Markdown, 16 HTML, 40 páginas de PDF (8 manuais de 5 páginas) e 40 linhas de
  FAQ.
- Os tamanhos variam muito: uma FAQ tem cerca de 230 caracteres; uma política, quase 5.000. Um vetor só por documento
  não é uma boa ideia, e é o assunto da próxima aula.
- Os dois documentos especiais aparecem com os metadados certos: é esse inventário que você deve olhar antes de
  indexar qualquer base nova.

## Resumo

- Um `Document` tem `page_content` (o texto, que vira vetor) e `metadata` (o resto). Metadados não entram no vetor,
  mas permitem **filtrar** (status, público) e **citar** (título, seção, página).
- Cada formato tem o seu carregador: front matter vira metadados; PDF vira um documento por página; HTML, só o
  `<article>`; CSV, um documento por linha.
- O que se descarta importa: menu e rodapé repetidos em todos os artigos aproximam vetores que deveriam ser diferentes.
- Problemas reais: PDF digitalizado (texto vazio, sem erro), HTML sem `<article>`, encoding errado (erro ou, pior,
  mojibake silencioso). Detecte, avise e corrija na origem.
- Os loaders do `langchain-community` estão em descontinuação; o projeto usa `pypdf`, `BeautifulSoup` e `csv`
  diretamente. `load_folder()` carrega a base inteira: 34 arquivos, 105 documentos.

## Próxima aula

Na aula 1.5 (`01_05_chunking_and_indexing`) vamos dividir esses 105 documentos em trechos, com o
`RecursiveCharacterTextSplitter`, o `MarkdownHeaderTextSplitter` e o cabeçalho de contexto, escolher o tamanho dos
trechos com um experimento e montar o índice com ids estáveis.